# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [ ]:
#@title 1.1 — Install (chạy một lần nếu môi trường chưa có dependencies)
# %pip -q install -r requirements.txt
# Sau khi cài xong, chọn Kernel > Restart Kernel trước khi chạy các cell tiếp theo.

In [15]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from dotenv import load_dotenv
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

PROJECT_ROOT = Path.cwd().resolve()
# Ưu tiên giá trị mới nhất trong .env thay vì token cũ từ process Jupyter.
load_dotenv(PROJECT_ROOT / ".env", override=True)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

if not HF_TOKEN or not HF_TOKEN.startswith("hf_"):
    raise ValueError("HF_TOKEN chưa được nạp đúng từ .env.")

DATA_PATH = DATA_DIR / "hackernoon_subset.csv"
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [16]:
from huggingface_hub import whoami
print(whoami(token=HF_TOKEN)["name"])

kacyyy


In [17]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = DATA_PATH

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 1_000_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Đang kết nối luồng dữ liệu (streaming)...
Đang ghi dữ liệu vào: D:\CODE\AITHUCCHIEN\LABS\K4-Track3-Lab19-GraphRAG-2A202601778-TranHoangKhoi\data\hackernoon_subset.csv


Đang tải (MB):  58%|█████▊    | 175.44/300 [01:18<00:55,  2.23MB/s]       
Exception ignored in: 'zmq.backend.cython._zmq.Frame.__dealloc__'
Traceback (most recent call last):
  File "zmq/backend/cython/_zmq.py", line 179, in zmq.backend.cython._zmq._check_rc
    PyErr_CheckSignals()
^^^^^^^^^^^
KeyboardInterrupt: 


In [18]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

# connect_neo4j()
# setup_graph_schema()

In [19]:
connect_neo4j()
setup_graph_schema()

✅ Neo4j connected.
✅ Schema ready.


In [20]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())

Exact dedup: 142,461 -> 123,246


Chunking: 100%|██████████| 1500/1500 [00:00<00:00, 100923.27it/s]


,chunk_id,article_id,title,published_date,text
0,c4d5799acafccf29c69d::c0000,c4d5799acafccf29c69d,Information Technology Policies,2023-07-15,One of the most important attributes of Information Technology (IT) policies is that they be readily available. That...
1,5e56702137be2b9f4d59::c0000,5e56702137be2b9f4d59,Kansas City''s Pepper Denver-based Notion combine to grow in InsurTech IoT sectors,2022-12-23,A #KansasCity tech company has combined with #Denver-based Notion to become a leader in IoT and smart home services.
2,aa2506534512eb6a40a5::c0000,aa2506534512eb6a40a5,Vehicle Rental Information,2023-06-30,Procurement and Contract Services would like to remind all faculty staff and students of the current State rental ca...
3,55a09cbc43c41ffb2dd9::c0000,55a09cbc43c41ffb2dd9,Transact Campus Partnership Lands Talkiatry Services on Campus Transact Apps,2023-09-05,The partnership will “provide students with access to quality psychiatric services and offers an accessible and affo...
4,198660b5d2a419ac68f2::c0000,198660b5d2a419ac68f2,Apple has been racketeering to protect its stolen services technology inventor accuses,2023-04-17,Inventor Patrick Racz claims his Smartflash technology is at the heart of the ... implied that — because his firm ma...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [21]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [23]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")
display(extraction_source[["chunk_id", "unresolved_mentions"]].head())

Coref: 100%|██████████| 80/80 [17:30<00:00, 13.14s/it]


,chunk_id,unresolved_mentions
0,c4d5799acafccf29c69d::c0000,"[we, our users', These]"
1,5e56702137be2b9f4d59::c0000,[]
2,aa2506534512eb6a40a5::c0000,[you]
3,55a09cbc43c41ffb2dd9::c0000,[]
4,198660b5d2a419ac68f2::c0000,[]


In [14]:
test_coref_df, test_usage = resolve_coref_batch(chunks_df.head(1))
display(test_coref_df)

RuntimeError
Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [24]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    return pd.DataFrame(triples), pd.DataFrame(errors)

raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
display(raw_triples_df.head())
display(extraction_errors_df.head())

NER+RE: 100%|██████████| 100/100 [17:08<00:00, 10.28s/it]


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Talkiatry,Company,PARTNERED_WITH,Transact,Company,55a09cbc43c41ffb2dd9::c0000,2023-09-05,The partnership will … Talkiatry said. Transact said Transact will provide partner institutions …,0.96
1,Patrick Racz,Person,FOUNDED,Patrick Racz's firm,Company,198660b5d2a419ac68f2::c0000,2023-04-17,Patrick Racz formed Patrick Racz's firm specifically ...,1.00
2,Patrick Racz,Person,DEVELOPED,Smartflash,Technology,198660b5d2a419ac68f2::c0000,2023-04-17,Inventor Patrick Racz claims Patrick Racz's Smartflash technology is at the heart of the ...,1.00
3,Claroty,Company,USES,ServiceNow,Company,127fbbd7afb3d21b8f46::c0000,2023-03-16,Claroty ... expanded its footprint within the ServiceNow ecosystem by releasing a new set of native integrations lev...,1.00
4,EarthSense,Company,PARTNERED_WITH,Telent Technology Services,Company,2abdfd676f2a64e7b735::c0000,2023-08-24,EarthSense ... will be supplying Telent Technology Services with EarthSense Zephyr air quality monitors,1.00


,start,error
0,176,Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k...
1,180,Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k...
2,184,Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k...
3,188,Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k...
4,192,Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k...


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [25]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok = merge_guard(names[i], names[j])
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
display(entity_resolution_audit_df.head(20))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2888.39it/s]


,type,left,right,similarity,decision
0,Technology,Sonos Era 100,Sonos Era 300,0.903168,MERGE_VECTOR


In [28]:
connect_neo4j()
bulk_insert_nodes(nodes_df, batch_size=100)
print("Đã insert nodes.")

✅ Neo4j connected.
Đã insert nodes.


In [29]:
connect_neo4j()
bulk_insert_edges(triples_df, batch_size=100)
print("Đã insert edges.")

✅ Neo4j connected.
Đã insert edges.


In [32]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)

In [34]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 89, 'edges': 55, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,1029cc5a61d01976b9615863,1Kosmos,Company,4
1,7495971aff6f73501d942b81,Worldcoin,Company,3
2,cf4c80448ad3907838936a79,Zipline,Company,3
3,3c1c26a78afb536a835368cc,OnePlus,Company,3
4,0e132222d1315530d6efca52,Google,Company,2
5,395a70ccd2569cdb32c62c5d,Jack O’Donnell,Person,2
6,c3bd261e79278b4d6b2b95db,Patrick Racz,Person,2
7,81b04d515dfc89bad6b43059,Bain & Company,Company,2
8,6dbfdfc5a4e8df3668a4dac3,Meta,Company,2
9,762ee40c148c0a9b8f1df5e0,Lava,Company,2


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [35]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches: 100%|██████████| 12/12 [00:18<00:00,  1.50s/it]

Flat vectors: 1500


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [38]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [39]:
build_entity_matcher(nodes_df)
print("Entity matcher ready:", len(entity_match_store))

Entity matcher ready: 89


In [40]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [41]:
test_query = "Which company partnered with Transact?"
debug_graph = retrieve_graph_context(test_query, return_debug=True)

print(debug_graph["context"])
print(debug_graph["diagnostics"])

Talkiatry [Company] -PARTNERED_WITH-> Transact [Company] | date=2023-09-05 | chunk=55a09cbc43c41ffb2dd9::c0000 | evidence=The partnership will … Talkiatry said. Transact said Transact will provide partner institutions …
{'matched_seeds': [{'id': '5bdf23135506d01440f0eb3d', 'name': 'Transact', 'type': 'Company'}], 'expanded_nodes': 2, 'collected_edges': 1, 'supernode_events': []}


In [42]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

In [44]:
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv(PROJECT_ROOT / ".env", override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq client updated.")

Groq client updated.


In [45]:
flat_result = answer_flat_rag(test_query)
graph_result = answer_graph_rag(test_query)

print("=== FLAT RAG ===")
print(flat_result["answer"])
print("Latency:", flat_result["latency_s"], "s")

print("\n=== HYBRID GRAPHRAG ===")
print(graph_result["answer"])
print("Latency:", graph_result["latency_s"], "s")

=== FLAT RAG ===
Talkiatry partnered with Transact 【55a09cbc43c41ffb2dd9::c0000】
Latency: 0.7122887999285012 s

=== HYBRID GRAPHRAG ===
Talkiatry partnered with Transact【chunk_id=55a09cbc43c41ffb2dd9::c0000】.
Latency: 0.7243066999362782 s


# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [46]:
#@title 4.1 — 5 câu Golden starter
GOLDEN_PATH = DATA_DIR / "golden_dataset.csv"

starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startups were founded by former Microsoft employees and later received investment from Google?",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Find a company invested in by a major technology company that also developed a named AI technology; identify both relations and dates.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
])

if Path(GOLDEN_PATH).exists():
    golden_df = pd.read_csv(GOLDEN_PATH)
else:
    golden_df = starter_golden.copy()
    golden_df.to_csv(GOLDEN_PATH, index=False)
    print(f"Đã tạo template Golden Dataset: {GOLDEN_PATH}")
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

Đã tạo template Golden Dataset: D:\CODE\AITHUCCHIEN\LABS\K4-Track3-Lab19-GraphRAG-2A202601778-TranHoangKhoi\data\golden_dataset.csv


,id,group,question,reference_answer,reference_evidence
0,G01,factoid,Who was the CEO of Hugging Face in 2023?,Clément Delangue,Validate against instructor dump.
1,G02,multi-hop,Which startups were founded by former Microsoft employees and later received investment from Google?,,TO_BE_FILLED_FROM_DATASET
2,G03,cross-doc,Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.,,TO_BE_FILLED_FROM_DATASET
3,G04,multi-hop,Find a company invested in by a major technology company that also developed a named AI technology; identify both re...,,TO_BE_FILLED_FROM_DATASET
4,G05,cross-doc,Identify one technology connected to the same company in at least two news chunks and summarize how the relationship...,,TO_BE_FILLED_FROM_DATASET


In [59]:
# Golden này được tạo từ các edge/chunk đang thật sự có trong Neo4j + FAISS.
# Không dùng bộ first5000 vì nó không cùng corpus random 1,500 bài của lab.
GOLDEN_PATH = DATA_DIR / "golden_dataset.csv"
golden_df = pd.read_csv(GOLDEN_PATH)

# Fail fast nếu ai thay Golden bằng evidence không thuộc Flat index hiện tại.
evidence_text = " ".join(golden_df["reference_evidence"].fillna(""))
golden_chunk_ids = set(re.findall(r"[0-9a-f]{20}::c\d{4}", evidence_text))
if "flat_store" in globals():
    missing = golden_chunk_ids - set(flat_store["chunk_id"])
    if missing:
        raise ValueError(f"Golden evidence not in Flat index: {sorted(missing)}")
print(f"Using aligned Golden: {len(golden_df)} questions, {len(golden_chunk_ids)} evidence chunks.")

validate_golden(golden_df)
display(golden_df[["id", "group", "question", "reference_answer"]])

Using aligned Golden: 5 questions, 5 evidence chunks.
✅ Golden Dataset valid.


,id,group,question,reference_answer
0,GAL-01,factoid,Which company partnered with Transact in the selected news corpus?,Talkiatry partnered with Transact.
1,GAL-02,multi-hop,Who leads Lava and which technology is Lava associated with?,Wen Miao leads Lava; Lava is associated with AI as an AI-powered fan-experience platform.
2,GAL-03,multi-hop,Which organization partnered with both Google and Temasek in the joint report?,Bain & Company partnered with both Google and Temasek in the joint report.
3,GAL-04,cross-doc,Across the August and October 2023 reports what two BlockID offerings were developed by 1Kosmos?,1Kosmos developed BlockID Identity orchestration and the BlockID platform.
4,GAL-05,cross-doc,How did 1Kosmos's reported BlockID offering change from August to October 2023?,The August report named BlockID Identity orchestration; the October report named the BlockID platform. The graph sho...


In [48]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [51]:
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / ".env", override=True)
JUDGE_PROVIDER = os.getenv("JUDGE_PROVIDER", "groq").lower()
JUDGE_MODEL = os.getenv("JUDGE_MODEL")

print("Judge provider:", JUDGE_PROVIDER)
print("Judge model:", JUDGE_MODEL)

Judge provider: groq
Judge model: openai/gpt-oss-120b


In [60]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = OUTPUT_DIR / "graphrag_eval_checkpoint.csv"

def run_evaluation(golden_df):
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)

✅ Golden Dataset valid.


Evaluation: 100%|██████████| 5/5 [01:37<00:00, 19.50s/it]


,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,GAL-01,factoid,Which company partnered with Transact in the selected news corpus?,Talkiatry partnered with Transact.,Talkiatry partnered with Transact 【chunk_id=55a09cbc43c41ffb2dd9::c0000】,"The provided excerpts do not contain any mention of a partnership involving Transact, so the information needed to a...",5,1,5,1,5,1,0.982163,1.001761,770,656,"The candidate correctly identifies Talkiatry as the company partnered with Transact, matching the reference answer e...","The reference clearly states that Talkiatry partnered with Transact, providing a direct answer. The candidate answer...",0
1,GAL-02,multi-hop,Who leads Lava and which technology is Lava associated with?,Wen Miao leads Lava; Lava is associated with AI as an AI-powered fan-experience platform.,"Lava is led by **Wen Miao**, who serves as its CEO, and the company is built around an **AI‑powered fan‑experience p...","Wen Miao leads Lava, and Lava is associated with the AI technology — it is an AI‑powered fan‑experience platform【chu...",5,5,5,5,5,5,0.821028,0.770199,761,705,The candidate correctly identifies Wen Miao as the leader of Lava and specifies that Lava is an AI‑powered fan‑exper...,"The candidate correctly identifies Wen Miao as the leader of Lava and states that Lava is associated with AI, matchi...",0
2,GAL-03,multi-hop,Which organization partnered with both Google and Temasek in the joint report?,Bain & Company partnered with both Google and Temasek in the joint report.,"The joint report was produced by **Google, Temasek, and Bain & Company**—so Bain & Company is the organization that ...",Bain & Company partnered with both Google and Temasek in the joint report 【cdb57fff932a8af9e4a5::c0000】.,5,5,5,5,5,5,0.617389,0.861187,742,783,"The candidate correctly identifies Bain & Company as the organization that partnered with both Google and Temasek, m...","The candidate directly answers the question by naming Bain & Company, exactly matching the reference answer. It incl...",0
3,GAL-04,cross-doc,Across the August and October 2023 reports what two BlockID offerings were developed by 1Kosmos?,1Kosmos developed BlockID Identity orchestration and the BlockID platform.,"The provided excerpts do not list the specific BlockID offerings that 1Kosmos developed in its August 2023 report, n...",The August 2023 report notes that 1Kosmos developed **BlockID Identity orchestration** [chunk=09836750d9cd56953ac9::...,1,5,1,5,1,5,7.312285,8.229504,870,917,"The reference clearly states the two offerings (BlockID Identity orchestration and BlockID platform), but the candid...",The candidate correctly lists both BlockID offerings—Identity orchestration from the August 2023 report and the plat...,0
4,GAL-05,cross-doc,How did 1Kosmos's reported BlockID offering change from August to October 2023?,The August report named BlockID Identity orchestration; the October report named the BlockID platform. The graph sho...,The provided sources include a Business Wire release from October 24 2023 that describes 1Kosmos’s BlockID platform ...,"In August 2023 the company was describing its offering as **“BlockID Identity orchestration,”** positioning it as a ...",1,5,1,5,1,5,3.089351,3.833312,829,998,The candidate answer fails to mention the August report's naming of BlockID Identity orchestration and thus does not...,The answer correctly identifies the August offering as BlockID Identity orchestration and the October offering as th...,0


In [61]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv(OUTPUT_DIR / "graphrag_eval_results.csv", index=False)
comparison_df.to_csv(OUTPUT_DIR / "graphrag_vs_flatrag_summary.csv", index=False)

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,1.000,5.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
1,cross-doc,Faithfulness,1.000,5.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
2,cross-doc,Multi-hop reasoning,1.000,5.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
3,cross-doc,Latency (s),5.201,6.031,Flat RAG thường rẻ/nhanh hơn.
4,cross-doc,Token usage,849.500,957.500,Flat RAG thường rẻ/nhanh hơn.
5,factoid,Comprehensiveness,5.000,1.000,Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.
6,factoid,Faithfulness,5.000,1.000,Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.
7,factoid,Multi-hop reasoning,5.000,1.000,Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.
8,factoid,Latency (s),0.982,1.002,Flat RAG thường rẻ/nhanh hơn.
9,factoid,Token usage,770.000,656.000,GraphRAG không đắt hơn trong sample này.


In [54]:
from pathlib import Path

for name in [
    "graphrag_eval_results.csv",
    "graphrag_vs_flatrag_summary.csv",
]:
    path = OUTPUT_DIR / name
    print(name, path.exists(), path)

graphrag_eval_results.csv True D:\CODE\AITHUCCHIEN\LABS\K4-Track3-Lab19-GraphRAG-2A202601778-TranHoangKhoi\outputs\graphrag_eval_results.csv
graphrag_vs_flatrag_summary.csv True D:\CODE\AITHUCCHIEN\LABS\K4-Track3-Lab19-GraphRAG-2A202601778-TranHoangKhoi\outputs\graphrag_vs_flatrag_summary.csv


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [56]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)

{'id': '1029cc5a61d01976b9615863', 'name': '1Kosmos', 'degree': 4} fetched= 4


,type,left,right,similarity,decision
0,Technology,Sonos Era 100,Sonos Era 300,0.903168,MERGE_VECTOR


High-similarity rejected pairs:


,type,left,right,similarity,decision


In [57]:
def build_extended_audit(raw_df, top_k=4, merge_threshold=0.90):
    mentions = []
    for r in raw_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    rows = []

    for typ in sorted(ALLOWED_NODE_TYPES):
        names = sorted({
            norm_space(name) for entity_type, name in mentions
            if entity_type == typ and norm_space(name)
        })
        if len(names) < 2:
            continue

        vectors = get_embedder().encode(
            names,
            normalize_embeddings=True,
            show_progress_bar=False
        ).astype("float32")

        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
        sims, neighbors = index.search(vectors, min(top_k, len(names)))

        for i, (scores, ids) in enumerate(zip(sims, neighbors)):
            for score, j in zip(scores, ids):
                if i >= j:
                    continue

                left, right = names[i], names[int(j)]
                if float(score) >= merge_threshold:
                    decision = (
                        "MERGE_VECTOR"
                        if merge_guard(left, right)
                        else "REJECT_GUARD"
                    )
                else:
                    decision = "REJECT_THRESHOLD"

                rows.append({
                    "type": typ,
                    "left": left,
                    "right": right,
                    "similarity": float(score),
                    "decision": decision,
                    "reason": (
                        "Passed vector threshold and lexical guard."
                        if decision == "MERGE_VECTOR"
                        else "High similarity but lexical guard rejected."
                        if decision == "REJECT_GUARD"
                        else "Below merge threshold 0.90; retained separately."
                    ),
                })

    return pd.DataFrame(rows)

extended_audit_df = build_extended_audit(raw_triples_df)

entity_resolution_audit_df = pd.concat(
    [entity_resolution_audit_df, extended_audit_df],
    ignore_index=True,
).drop_duplicates(subset=["type", "left", "right", "decision"])

print("Audit rows:", len(entity_resolution_audit_df))
display(
    entity_resolution_audit_df
    .sort_values("similarity", ascending=False)
    .head(30)
)

Audit rows: 130


,type,left,right,similarity,decision,reason
0,Technology,Sonos Era 100,Sonos Era 300,0.903168,MERGE_VECTOR,NaN
4,Company,3M,3M Co,0.820720,REJECT_THRESHOLD,Below merge threshold 0.90; retained separately.
95,Technology,BlockID Identity orchestration,BlockID platform,0.653893,REJECT_THRESHOLD,Below merge threshold 0.90; retained separately.
90,Technology,AI,Bitcoin 360 AI,0.549836,REJECT_THRESHOLD,Below merge threshold 0.90; retained separately.
91,Technology,AI,autonomous vehicles,0.533957,REJECT_THRESHOLD,Below merge threshold 0.90; retained separately.
87,Person,Michael Cichon,Michael Roman,0.522819,REJECT_THRESHOLD,Below merge threshold 0.90; retained separately.
92,Technology,AI,intelligent software,0.519379,REJECT_THRESHOLD,Below merge threshold 0.90; retained separately.
13,Company,Apple,Intel,0.513554,REJECT_THRESHOLD,Below merge threshold 0.90; retained separately.
71,Person,Brian Robinson,Mike Engle,0.503446,REJECT_THRESHOLD,Below merge threshold 0.90; retained separately.
16,Company,Bain & Company,HDFC Asset Management Company,0.490686,REJECT_THRESHOLD,Below merge threshold 0.90; retained separately.


In [58]:
entity_resolution_audit_df.to_csv(
    OUTPUT_DIR / "entity_resolution_audit.csv",
    index=False
)

## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [ ]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()

In [ ]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau